In [2]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [3]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'
MINCUBESAMPLES = 100
TARGETNAME = 'sr_all_eq'

SRFUNCTIONS = {
    'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
    'sqrt':np.sqrt,'exp':np.exp,'log':np.log,'abs':np.abs,
    'sin':np.sin,'cos':np.cos,'max':np.maximum,'min':np.minimum,
    '_safepow':lambda a,b:np.abs(a)**b}

import re
def _prepare_form(form):
    return re.sub(r'(\w+)\^(\w+)',r'_safepow(\1,\2)',form)

def eval_form(form,columns,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    ns.update(columns)
    ns.update(constants)
    out = eval(_prepare_form(form),ns)
    if np.ndim(out)==0:
        n = len(next(v for v in columns.values() if hasattr(v,'__len__')))
        out = np.full(n,float(out))
    return np.asarray(out,dtype=float)

In [4]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS['tp_mean']
STD  = STATS['tp_std']
ZMIN = (0.0 - MEAN) / STD

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds['dsig'].values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lfraw = flat('lf')
    shfraw = flat('shf')
    lhfraw = flat('lhf')
    blraw = flat('bl') if 'bl' in ds else np.zeros(ntime*ds.sizes['lat']*ds.sizes['lon'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsraw = ds['tp'].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)
meankernel = np.mean(kernels,axis=0)
weighted = fields * meankernel[None,:,:] * dsig[None,None,:]
if surfmask is not None:
    weighted = weighted * surfmask[:,None,:]
integrals = weighted.sum(axis=2)
rhraw,thetaeraw,thetaestarraw = integrals[:,0],integrals[:,1],integrals[:,2]

valid = np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw) & np.isfinite(obsraw)
rh,thetae,thetaestar = rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf,shf,lhf = lfraw[valid],shfraw[valid],lhfraw[valid]
bl = blraw[valid]
obs = obsraw[valid]
landmask  = lf > 0.5
oceanmask = lf < 0.5
print(f'Loaded {valid.sum():,} valid samples ({landmask.sum():,} land, {oceanmask.sum():,} ocean)')

Loaded 1,437,408 valid samples (428,352 land, 1,009,056 ocean)


In [5]:
regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in regdf.iterrows()}
SRMODELS = CONFIGS['experiments']['sr']['optimizedeqs']
ORDER  = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}
COLORS = {name:SRMODELS[name]['color'] for name in ORDER}

def get_columns(**overrides):
    cols = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
            'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    cols.update(overrides)
    for eqname,entry in REGISTRY.items():
        if eqname in overrides:
            continue
        cols[eqname] = eval_form(entry['form'],cols,entry['constants'])
    return cols

def predict_eq(name,columns):
    entry = REGISTRY[name]
    raw = eval_form(entry['form'],columns,entry['constants'])
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

cols = get_columns()
pred = predict_eq(TARGETNAME,cols)

r2all   = 1 - np.mean((pred - obs)**2) / np.var(obs)
r2land  = 1 - np.mean((pred[landmask] - obs[landmask])**2) / np.var(obs[landmask])
r2ocean = 1 - np.mean((pred[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])
print(f'{LABELS[TARGETNAME]} (unconstrained):')
print(f'  R\u00b2 all={r2all:.4f}  land={r2land:.4f}  ocean={r2ocean:.4f}')

SR-ALL (unconstrained):
  R² all=0.4149  land=0.2926  ocean=0.5089


In [6]:
BASEVARS = ['rh','thetae','thetaestar','lf','shf','lhf','bl']
NCUBES = [3,4,5,6,7]

CONSTRAINTS = {
    'PC2':{
        'label':r'$\partial P/\partial \widehat{\mathrm{RH}} \geq 0$',
        'target_var':'rh',
        'expected_sign':1,
        'other_vars':['thetae','thetaestar']},
    'PC3':{
        'label':r'$\partial P/\partial \widehat{\theta_e} \geq 0$',
        'target_var':'thetae',
        'expected_sign':1,
        'other_vars':['rh','thetaestar']},
    'PC4':{
        'label':r'$\partial P/\partial \widehat{\theta_e^*} \leq 0$',
        'target_var':'thetaestar',
        'expected_sign':-1,
        'other_vars':['rh','thetae']}}

def cube_monotonicity_test(name,target_var,expected_sign,other_vars,ncubes,mask=None):
    cols = get_columns()
    targetvals = cols[target_var]
    othervalslist = [cols[v] for v in other_vars]
    nother = len(other_vars)
    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]
    nsatisfied,ntested = 0,0
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]
    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        nsel = sel.sum()
        x = targetvals[sel]
        overrides = {k:cols[k][sel] for k in BASEVARS}
        overrides[target_var] = x
        for v,binarr,edgearr in zip(other_vars,bins,edges):
            ci = cidx
            for j in range(nother-1,-1,-1):
                if other_vars[j] == v:
                    bi = ci % ncubes
                    break
                ci //= ncubes
            midpoint = 0.5 * (edgearr[bi] + edgearr[bi+1])
            overrides[v] = np.full(nsel,midpoint)
        cubecols = get_columns(**overrides)
        p = predict_eq(name,cubecols)
        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,p) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expected_sign >= 0 and slope >= 0) or (expected_sign < 0 and slope <= 0):
            nsatisfied += 1
    return nsatisfied,ntested

results = {}
for pcname,pc in CONSTRAINTS.items():
    results[pcname] = {}
    for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
        pcts = []
        for n in NCUBES:
            sat,tot = cube_monotonicity_test(TARGETNAME,pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask=mask)
            pcts.append(sat/max(tot,1)*100)
        results[pcname][region] = np.mean(pcts)
    print(f'{pcname} ({pc["label"]}): Land={results[pcname]["Land"]:.1f}% '
          f'Ocean={results[pcname]["Ocean"]:.1f}% All={results[pcname]["All"]:.1f}%')

ENFORCE_THRESHOLD = 90.0
enforce_regions = {}
for pcname in CONSTRAINTS:
    enforce_regions[pcname] = {}
    for region in ['Land','Ocean']:
        enforce_regions[pcname][region] = results[pcname][region] >= ENFORCE_THRESHOLD

print()
for pcname in CONSTRAINTS:
    land_str = 'ENFORCE' if enforce_regions[pcname]['Land'] else 'skip'
    ocean_str = 'ENFORCE' if enforce_regions[pcname]['Ocean'] else 'skip'
    print(f'{pcname}: land={land_str}, ocean={ocean_str}')

PC2 ($\partial P/\partial \widehat{\mathrm{RH}} \geq 0$): Land=95.3% Ocean=99.2% All=97.7%
PC3 ($\partial P/\partial \widehat{\theta_e} \geq 0$): Land=90.9% Ocean=87.8% All=91.8%
PC4 ($\partial P/\partial \widehat{\theta_e^*} \leq 0$): Land=100.0% Ocean=100.0% All=100.0%

PC2: land=ENFORCE, ocean=ENFORCE
PC3: land=ENFORCE, ocean=skip
PC4: land=ENFORCE, ocean=ENFORCE


In [ ]:
entry = REGISTRY[TARGETNAME]
atmentry = REGISTRY['sr_atm_eq']

print(f'SR-ALL form:     {entry["form"]}')
print(f'SR-ALL constants: {entry["constants"]}')
print(f'SR-ATM form:     {atmentry["form"]}')
print(f'SR-ATM constants: {atmentry["constants"]}')
print()

# Denote the SR-ATM constants as a_atm, b_atm, c_atm
# and the SR-ALL correction constants as a, b, c.
#
# SR-ALL = a_atm * cube(max(rh, thetae - b_atm*thetaestar - c_atm))
#        + (thetae + a*shf) * cube(b - lf)
#        + c
#
# The max() creates two branches:
#   RH branch:       rh >= thetae - b_atm*thetaestar - c_atm
#   Buoyancy branch: rh <  thetae - b_atm*thetaestar - c_atm
#
# --- PC2: dP/d(rh) >= 0 ---
#   RH branch:       3*a_atm*rh^2                       >= 0 if a_atm > 0  CHECK
#   Buoyancy branch: 0                                   >= 0              CHECK
#
# --- PC3: dP/d(thetae) >= 0 ---
#   RH branch:       cube(b - lf)                        SIGN DEPENDS ON b VS lf
#   Buoyancy branch: 3*a_atm*(thetae-b_atm*thetaestar-c_atm)^2 + cube(b-lf)
#                                                         FIRST TERM >= 0, SECOND UNCERTAIN
#
# --- PC4: dP/d(thetaestar) <= 0 ---
#   RH branch:       0                                   <= 0              CHECK
#   Buoyancy branch: -b_atm * 3*a_atm*(...)^2            <= 0 if a_atm,b_atm > 0  CHECK
#
# CONCLUSION: PC3 is the only analytically uncertain constraint.
# The violation comes from cube(b - lf), which is NEGATIVE when lf > b.

print('Verify a_atm > 0 and b_atm > 0 (required for PC2 and PC4):')
print(f'  a_atm = {atmentry["constants"]["a"]:.4f}  ({"OK" if atmentry["constants"]["a"] > 0 else "PROBLEM"})')
print(f'  b_atm = {atmentry["constants"]["b"]:.4f}  ({"OK" if atmentry["constants"]["b"] > 0 else "PROBLEM"})')
print()
print(f'SR-ALL correction constant b = {entry["constants"]["b"]:.4f}')
print(f'  Over ocean (lf ~ 0): cube({entry["constants"]["b"]:.2f} - 0) = {(entry["constants"]["b"])**3:.4f}')
print(f'  Over land  (lf ~ 1): cube({entry["constants"]["b"]:.2f} - 1) = {(entry["constants"]["b"] - 1)**3:.4f}')
print()
print('PC3 violation mechanism:')
print('  When lf > b, cube(b-lf) < 0, making dP/d(thetae) < 0 in the RH branch.')
print('  In the buoyancy branch, the positive 3*a_atm*(...)^2 term can compensate,')
print('  but not always — hence partial violations.')

In [ ]:
# Following Grundner et al. (2024): structurally modify the equation form
# so that all three physical constraints are satisfied ANALYTICALLY,
# then re-optimize constants via L-BFGS-B.
#
# The fix: replace cube(b - lf) with cube(max(b - lf, 0)).
#
#   cube(max(b - lf, 0)) >= 0 always, because max(b-lf,0) >= 0 and cube is monotone.
#
# This guarantees dP/d(thetae) >= 0 in both branches:
#   RH branch:       cube(max(b-lf,0)) >= 0                              CHECK
#   Buoyancy branch: 3*a_atm*(...)^2 + cube(max(b-lf,0)) >= 0           CHECK
#                    (sum of two non-negative terms)
#
# PC2 and PC4 are unaffected (the correction term does not involve rh or thetaestar).
#
# Trade-off: over land where lf > b, the correction term vanishes (clipped to zero),
# so the equation reduces to sr_atm_eq + c. The optimizer compensates by adjusting
# constants, and the sr_atm_eq base already captures the dominant thermodynamic signal.

PCFORM = 'sr_atm_eq+(thetae+a*shf)*cube(max(b-lf,0))+c'
PCNAME = 'sr_all_pc_eq'
PCLABEL = 'SR-ALL-PC'

print(f'Original:    {entry["form"]}')
print(f'Constrained: {PCFORM}')
print(f'Modification: cube(b-lf) -> cube(max(b-lf,0))')
print()
print(f'Initial constants from SR-ALL: {entry["constants"]}')

In [ ]:
from scipy.optimize import minimize

y = (np.log1p(obs) - MEAN) / STD
cols = get_columns()

constantnames = ['a','b','c']
init_all = entry['constants']
initparams = np.array([init_all[c] for c in constantnames])

def objective(params):
    constants = dict(zip(constantnames,params))
    raw = eval_form(PCFORM,cols,constants)
    pred = ZMIN + np.maximum(raw,0.0)
    return float(np.mean((pred - y)**2))

nrestarts = 50
rng = np.random.default_rng(42)
allresults = []
allinits = [initparams] + [rng.uniform(-5,5,len(constantnames)) for _ in range(nrestarts-1)]
for i,x0 in enumerate(allinits):
    res = minimize(objective,x0,method='L-BFGS-B',options={'maxiter':10000,'ftol':1e-14,'gtol':1e-10})
    allresults.append(res)
bestres = min(allresults,key=lambda r:r.fun)
pcconstants = dict(zip(constantnames,bestres.x))
pcconstants_rounded = {k:round(float(v),2) for k,v in pcconstants.items()}

print(f'Optimized on {SPLIT} split ({valid.sum():,} samples, {nrestarts} restarts)')
print(f'  Constants (raw):     {", ".join(f"{k}={v:.6f}" for k,v in pcconstants.items())}')
print(f'  Constants (rounded): {", ".join(f"{k}={v:.2f}" for k,v in pcconstants_rounded.items())}')
print(f'  MSE (z-space): {bestres.fun:.6f}  (SR-ALL unconstrained: {objective(initparams):.6f})')
print()
print(f'NOTE: These constants are optimized on the {SPLIT} split only.')
print('For final constants, run on train+valid via the pipeline:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq')

In [ ]:
def predict_pc(columns):
    raw = eval_form(PCFORM,columns,pcconstants_rounded)
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

def cube_monotonicity_test_pc(target_var,expected_sign,other_vars,ncubes,mask=None):
    cols = get_columns()
    targetvals = cols[target_var]
    othervalslist = [cols[v] for v in other_vars]
    nother = len(other_vars)
    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]
    nsatisfied,ntested = 0,0
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]
    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        nsel = sel.sum()
        x = targetvals[sel]
        overrides = {k:cols[k][sel] for k in BASEVARS}
        overrides[target_var] = x
        for v,binarr,edgearr in zip(other_vars,bins,edges):
            ci = cidx
            for j in range(nother-1,-1,-1):
                if other_vars[j] == v:
                    bi = ci % ncubes
                    break
                ci //= ncubes
            midpoint = 0.5 * (edgearr[bi] + edgearr[bi+1])
            overrides[v] = np.full(nsel,midpoint)
        cubecols = get_columns(**overrides)
        p = predict_pc(cubecols)
        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,p) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expected_sign >= 0 and slope >= 0) or (expected_sign < 0 and slope <= 0):
            nsatisfied += 1
    return nsatisfied,ntested

pcresults = {}
for pcname,pc in CONSTRAINTS.items():
    pcresults[pcname] = {}
    for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
        pcts = []
        for n in NCUBES:
            sat,tot = cube_monotonicity_test_pc(pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask=mask)
            pcts.append(sat/max(tot,1)*100)
        pcresults[pcname][region] = np.mean(pcts)

print(f'Constraint satisfaction: SR-ALL (unconstrained) vs SR-ALL-PC (constrained)')
print(f'{"":>6} {"":>38} {"SR-ALL":>10} {"SR-ALL-PC":>10} {"Delta":>8}')
for pcname in CONSTRAINTS:
    label = CONSTRAINTS[pcname]['label']
    for region in ['Land','Ocean','All']:
        old = results[pcname][region]
        new = pcresults[pcname][region]
        delta = new - old
        tag = f'{pcname} {region}'
        print(f'{tag:>18} {label:>22}  {old:>8.1f}%  {new:>8.1f}%  {delta:>+7.1f}%')

In [ ]:
predpc = predict_pc(cols)

r2pc_all   = 1 - np.mean((predpc - obs)**2) / np.var(obs)
r2pc_land  = 1 - np.mean((predpc[landmask] - obs[landmask])**2) / np.var(obs[landmask])
r2pc_ocean = 1 - np.mean((predpc[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])

print(f'Accuracy comparison ({SPLIT} split):')
print(f'{"Model":<14} {"R² all":>8} {"R² land":>9} {"R² ocean":>10}')
print(f'{"SR-ALL":<14} {r2all:>8.4f} {r2land:>9.4f} {r2ocean:>10.4f}')
print(f'{"SR-ALL-PC":<14} {r2pc_all:>8.4f} {r2pc_land:>9.4f} {r2pc_ocean:>10.4f}')
print(f'{"Delta":<14} {r2pc_all-r2all:>+8.4f} {r2pc_land-r2land:>+9.4f} {r2pc_ocean-r2ocean:>+10.4f}')

mse_all   = np.mean((pred - obs)**2)
mse_land  = np.mean((pred[landmask] - obs[landmask])**2)
mse_ocean = np.mean((pred[oceanmask] - obs[oceanmask])**2)
msepc_all   = np.mean((predpc - obs)**2)
msepc_land  = np.mean((predpc[landmask] - obs[landmask])**2)
msepc_ocean = np.mean((predpc[oceanmask] - obs[oceanmask])**2)

print()
print(f'{"Model":<14} {"MSE all":>9} {"MSE land":>10} {"MSE ocean":>11}')
print(f'{"SR-ALL":<14} {mse_all:>9.4f} {mse_land:>10.4f} {mse_ocean:>11.4f}')
print(f'{"SR-ALL-PC":<14} {msepc_all:>9.4f} {msepc_land:>10.4f} {msepc_ocean:>11.4f}')

In [ ]:
PCLABELS = {k:v['label'] for k,v in CONSTRAINTS.items()}
rows = []
for model,label,r2,constresults in [
    (TARGETNAME,'SR-ALL',[r2all,r2land,r2ocean],results),
    (PCNAME,'SR-ALL-PC',[r2pc_all,r2pc_land,r2pc_ocean],pcresults)]:
    row = {'Model':label,'R² all':r2[0],'R² land':r2[1],'R² ocean':r2[2]}
    for pcname in CONSTRAINTS:
        row[PCLABELS[pcname]] = constresults[pcname]['All']
    rows.append(row)

summdf = pd.DataFrame(rows).set_index('Model')
summdf.style.format({
    'R² all':'{:.4f}','R² land':'{:.4f}','R² ocean':'{:.4f}',
    **{PCLABELS[k]:'{:.1f}%' for k in CONSTRAINTS}
}).set_caption(f'Summary: accuracy and physical constraint satisfaction ({SPLIT} split)')

In [ ]:
print('New entry for configs.json (experiments.sr.optimizedeqs):')
print()
print(json.dumps({PCNAME:{
    'runfrom':'sr_all',
    'refcomplexity':None,
    'form':PCFORM,
    'init':{k:round(float(v),2) for k,v in pcconstants.items()},
    'color':'#8B0000',
    'description':PCLABEL}},indent=4))
print()
print('To optimize final constants on train+valid:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq')
print()
print('To generate predictions:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq --predict-only --splits test')